In [ ]:

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import shap


# ---------- 0. 路径 ----------
DATA_FILE   = "Beautiful_scored.xlsx"
UNSCORED    = "images.xlsx"
OUT_DIR     = Path("D:\Desktop\output_Beautiful2")
OUT_DIR.mkdir(exist_ok=True)

PREDICT_XLSX = OUT_DIR / "images_predicted_Beautiful.xlsx"
BSWARM_PNG   = OUT_DIR / "beautiful_shap_beeswarm.png"
BAR_PNG      = OUT_DIR / "beautiful_shap_bar.png"

# ---------- 1. 常量 ----------
TARGET = "Beautiful"
FEATS  = ['Road','Building','Pole Group','Indicator','Vegetation','Sky',
          'Person','Car','Motorcycle','Bicycle','Clothes','Trash Can',
          'Riverway','Signboard','Air Conditioner Condenser','Festival Elements']

# ---------- 2. 读取已打分数据 ----------
df = pd.read_excel(DATA_FILE)
df[TARGET] = df[TARGET].round(5)

X = df[FEATS].copy()         
y = df[TARGET].values

# ---------- 3. 划分 & 训练----------
X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=42)

rf = RandomForestRegressor(
        n_estimators     = 150,
        max_depth        = 6,
        min_samples_leaf = 8,
        min_samples_split= 12,
        max_features     = "sqrt",
        random_state     = 42,
        n_jobs           = -1
     ).fit(X_tr, y_tr)

# ---------- 4. 单调校准
iso = IsotonicRegression(
        y_min=y.min(), y_max=y.max(),
        increasing=True, out_of_bounds="clip").fit(rf.predict(X), y)

def _m(t,p): return r2_score(t,p), mean_squared_error(t,p,squared=False)
cal_tr = iso.transform(rf.predict(X_tr)); cal_te = iso.transform(rf.predict(X_te))
r2_all, rmse_all = _m(y,     iso.transform(rf.predict(X)))

print(f"Train R²={_m(y_tr, cal_tr)[0]:.3f} | RMSE={_m(y_tr, cal_tr)[1]:.3f}")
print(f"Test  R²={_m(y_te, cal_te)[0]:.3f} | RMSE={_m(y_te, cal_te)[1]:.3f}")
print(f"Overall R² = {r2_all:.3f} | RMSE = {rmse_all:.3f}")


# ---------- 5. SHAP ----------
expl = shap.TreeExplainer(rf); sv = expl.shap_values(X)

plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(sv, X, feature_names=FEATS, show=False)
plt.title("SHAP Beeswarm — Beautiful")
plt.savefig(BSWARM_PNG, bbox_inches="tight",dpi=300); plt.close()

plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(sv, X, feature_names=FEATS, plot_type="bar", show=False)
plt.title("Feature Importance (mean |SHAP value|)")
plt.savefig(BAR_PNG, bbox_inches="tight",dpi=300); plt.close()

# ---------- 6. 预测未打分样本 ----------
df_new = pd.read_excel(UNSCORED)
X_new  = df_new[FEATS].copy()
df_new[TARGET] = iso.transform(rf.predict(X_new)).round(5)
df_new.to_excel(PREDICT_XLSX, index=False, float_format="%.5f")

print("✔ 预测文件:", PREDICT_XLSX)
print("✔ SHAP 图:", BSWARM_PNG, BAR_PNG)


Train R²=0.893 | RMSE=0.195
Test  R²=0.858 | RMSE=0.226
Overall R² = 0.886 | RMSE = 0.201
✔ 预测文件: D:\Desktop\output_Beautiful2\images_predicted_Beautiful.xlsx
✔ SHAP 图: D:\Desktop\output_Beautiful2\beautiful_shap_beeswarm.png D:\Desktop\output_Beautiful2\beautiful_shap_bar.png
